[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/03_efficient_deployment.ipynb)

# 03. Efficient Deployment: Ship Your Multimodal Model

**This notebook covers:**
- ONNX export — run anywhere, fast inference
- Post-training quantization (int8, int4)
- Model optimization techniques
- Benchmarking: measure speed and memory
- Gradio demo — deploy in 5 lines

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/05_Advanced_Topics")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time
import os
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# Deployment options overview

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('Deployment Options for Multimodal Models', fontsize=18, fontweight='bold', pad=20)

draw_architecture_block(ax, 3, 6, 4, 0.8, 'Trained PyTorch Model', '#3498DB')

options = [
    (2, 4, 'ONNX Export\n(cross-platform)', '#E74C3C'),
    (6, 4, 'TorchScript\n(PyTorch native)', '#F39C12'),
    (10, 4, 'Quantization\n(int8/int4)', '#2ECC71'),
    (2, 2, 'ONNX Runtime\n(CPU/GPU, fast)', '#C0392B'),
    (6, 2, 'torch.compile\n(2x speedup)', '#E67E22'),
    (10, 2, 'vLLM / TGI\n(serving)', '#27AE60'),
    (6, 0.5, 'Gradio / FastAPI\n(Demo / API)', '#9B59B6'),
]

for x, y, label, color in options:
    draw_architecture_block(ax, x, y, 3.5, 0.8, label, color, fontsize=9)

draw_arrow(ax, (2, 5.5), (2, 4.5))
draw_arrow(ax, (3, 5.5), (6, 4.5))
draw_arrow(ax, (4, 5.5), (10, 4.5))
draw_arrow(ax, (2, 3.5), (2, 2.5))
draw_arrow(ax, (6, 3.5), (6, 2.5))
draw_arrow(ax, (10, 3.5), (10, 2.5))
draw_arrow(ax, (6, 1.5), (6, 1.0))

plt.tight_layout()
plt.savefig('../assets/deployment_options.png', dpi=150, bbox_inches='tight')
plt.show()

## 1. ONNX Export

In [ ]:
# Create a small model to export

class SmallImageEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.proj = nn.Linear(64, embed_dim)

    def forward(self, x):
        return F.normalize(self.proj(self.features(x)), dim=-1)


model = SmallImageEncoder(embed_dim=128)
model.eval()
dummy = torch.randn(1, 3, 32, 32)

# Export to ONNX
onnx_path = '../assets/image_encoder.onnx'
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['image'],
    output_names=['embedding'],
    dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)

file_size = os.path.getsize(onnx_path) / 1024
print(f"ONNX model saved: {onnx_path}")
print(f"File size: {file_size:.1f} KB")

# Verify with ONNX Runtime
try:
    import onnxruntime as ort
    session = ort.InferenceSession(onnx_path)
    result = session.run(None, {'image': dummy.numpy()})
    print(f"ONNX output shape: {result[0].shape}")
    
    # Verify outputs match
    with torch.no_grad():
        torch_out = model(dummy).numpy()
    diff = np.abs(torch_out - result[0]).max()
    print(f"Max difference PyTorch vs ONNX: {diff:.8f}")
except ImportError:
    print("Install onnxruntime: pip install onnxruntime")

## 2. Post-Training Quantization

In [ ]:
# PyTorch dynamic quantization (CPU)

model_fp32 = SmallImageEncoder(128)
model_fp32.eval()

# Dynamic quantization (quantize linear layers to int8)
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32, {nn.Linear}, dtype=torch.qint8
)

# Compare sizes
def get_model_size(model):
    torch.save(model.state_dict(), '/tmp/temp_model.pt')
    size = os.path.getsize('/tmp/temp_model.pt')
    os.remove('/tmp/temp_model.pt')
    return size

fp32_size = get_model_size(model_fp32)
int8_size = get_model_size(model_int8)

print(f"FP32 model: {fp32_size/1024:.1f} KB")
print(f"INT8 model: {int8_size/1024:.1f} KB")
print(f"Reduction:  {(1-int8_size/fp32_size)*100:.1f}%")

In [ ]:
# Benchmark: speed comparison

def benchmark(model, input_tensor, n_runs=100, label='Model'):
    model.eval()
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            model(input_tensor)
    
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = time.perf_counter()
            model(input_tensor)
            times.append(time.perf_counter() - start)
    
    avg = np.mean(times) * 1000
    std = np.std(times) * 1000
    print(f"{label:20s}: {avg:.2f} +/- {std:.2f} ms")
    return times


dummy = torch.randn(1, 3, 32, 32)
t_fp32 = benchmark(model_fp32, dummy, label='FP32')
t_int8 = benchmark(model_int8, dummy, label='INT8 (quantized)')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([np.array(t_fp32)*1000, np.array(t_int8)*1000], 
           labels=['FP32', 'INT8'], patch_artist=True,
           boxprops=[dict(facecolor='#3498DB', alpha=0.7), 
                     dict(facecolor='#2ECC71', alpha=0.7)])
ax.set_ylabel('Latency (ms)')
ax.set_title('Inference Speed: FP32 vs INT8 Quantized', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

speedup = np.mean(t_fp32) / np.mean(t_int8)
print(f"\nSpeedup: {speedup:.2f}x")

## 3. torch.compile (PyTorch 2.0+)

The simplest optimization — one line of code for up to 2x speedup.

In [ ]:
# torch.compile example
model_eager = SmallImageEncoder(128)
model_eager.eval()

try:
    model_compiled = torch.compile(model_eager, mode='reduce-overhead')
    
    dummy = torch.randn(1, 3, 32, 32)
    # Warmup compile
    with torch.no_grad():
        for _ in range(5):
            model_compiled(dummy)
    
    t_eager = benchmark(model_eager, dummy, label='Eager')
    t_compiled = benchmark(model_compiled, dummy, label='Compiled')
    
    speedup = np.mean(t_eager) / np.mean(t_compiled)
    print(f"\ntorch.compile speedup: {speedup:.2f}x")
except Exception as e:
    print(f"torch.compile not available: {e}")
    print("Works with PyTorch 2.0+ and Linux")

## 4. Quick Demo with Gradio

Deploy your model as an interactive web app in 5 lines.

In [ ]:
# Gradio demo template
gradio_code = '''
import gradio as gr
import torch
from PIL import Image
import open_clip

# Load model
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model.eval()

def classify_image(image, text_labels):
    """Zero-shot classify an image given text labels."""
    labels = [l.strip() for l in text_labels.split(",")]
    
    img = preprocess(image).unsqueeze(0)
    text = tokenizer(labels)
    
    with torch.no_grad():
        img_feat = model.encode_image(img)
        txt_feat = model.encode_text(text)
        img_feat /= img_feat.norm(dim=-1, keepdim=True)
        txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
        probs = (100 * img_feat @ txt_feat.T).softmax(dim=-1)[0]
    
    return {label: float(prob) for label, prob in zip(labels, probs)}

demo = gr.Interface(
    fn=classify_image,
    inputs=[
        gr.Image(type="pil"),
        gr.Textbox(label="Labels (comma-separated)", 
                   value="cat, dog, car, house, tree")
    ],
    outputs=gr.Label(num_top_classes=5),
    title="Zero-Shot Image Classification with CLIP",
)

demo.launch()
'''

print("Save this as 'app.py' and run with: python app.py")
print("="*50)
print(gradio_code)

## Deployment Cheat Sheet

| Method | Speedup | Memory | Effort | Best For |
|--------|---------|--------|--------|----------|
| **ONNX** | 1.5-3x | Same | Easy | Cross-platform, edge |
| **torch.compile** | 1.2-2x | Same | 1 line | Quick wins |
| **INT8 quantization** | 1.5x | 0.5x | Easy | CPU deployment |
| **INT4 (GPTQ/AWQ)** | 2x | 0.25x | Medium | LLM serving |
| **TensorRT** | 2-5x | Varies | Medium | NVIDIA GPUs |
| **Gradio** | N/A | N/A | Easy | Quick demos |
| **vLLM** | 3-10x | Optimized | Medium | LLM production |

---

## Congratulations!

You've completed the entire Multimodal Learning course! Here's what you learned:

1. **Foundations:** Modalities, encoders, fusion strategies
2. **Models:** CLIP, captioning, VQA — built from scratch
3. **Training:** Contrastive learning, multi-objective, full pipeline
4. **Finetuning:** LoRA, QLoRA, adapters — for low compute
5. **Advanced:** LLaVA, audio+video, efficient deployment

**Next steps:**
- Finetune CLIP on your own domain data
- Build a LLaVA-style model with QLoRA
- Deploy with Gradio or ONNX
- Explore HuggingFace model hub for pretrained multimodal models